In [1]:
#Messy data
raw_products = [
    {"name": "Wireless Mouse",  "price": "$25.99",   "category": "Electronics"},
    {"name": " wireless mouse ", "price": "25.99",   "category": "electronics"},  # duplicate
    {"name": "USB Keyboard",    "price": 45,         "category": "Electronics"},
    {"name": "Monitor 24in",    "price": "$180.00",  "category": "ELECTRONICS"},
    {"name": "Webcam",          "price": "59.5",     "category": "Electronics"},
    {"name": "Headphones",      "price": "89",       "category": "Electronics"},
    {"name": "Laptop",          "price": "$1,899",   "category": "Electronics"},
    {"name": "Smart Watch",     "price": 150,        "category": "Electronics"},
    {"name": "Power Bank",      "price": "30",       "category": "Electronics"},
    {"name": "Notebook A5",     "price": "120",      "category": "Stationery"},
    {"name": "Gel Pen",         "price": "15",       "category": "stationery"},
    {"name": "Stapler",         "price": None,       "category": "Stationery"},   # price missing
    {"name": "Marker Set",      "price": "N/A",      "category": "Stationery"},   # bad price
    {"name": "Desk Lamp",       "price": 35,         "category": ""},             # category missing
    {"name": "Water Bottle",    "price": "  12.5 ",  "category": "Kitchen"},
    {"name": "Lunch Box",       "price": 18,         "category": "Kitchen"},
    {"name": "",                "price": 10,         "category": "Misc"},          # name missing
]

In [2]:
#Small helpers
import re

def clean_price(v):
    """Kisi bhi type ki price -> float, warna None."""
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        return float(v)
    m = re.search(r"\d[\d,]*\.?\d*", str(v or ""))
    return float(m.group().replace(",", "")) if m else None

def clean_name(v):
    return " ".join(str(v or "").split())            # extra spaces hatao

def clean_category(v):
    c = str(v or "").strip()
    return c.title() if c else "Uncategorized"

In [3]:
#Cleaning + grouping + top 5 (sab comprehensions)
# 1) Normalize
normalized = [
    {"name": clean_name(r["name"]),
     "price": clean_price(r["price"]),
     "category": clean_category(r["category"])}
    for r in raw_products
]

# 2) Name ya price missing wali rows drop
valid = [p for p in normalized if p["name"] and p["price"] is not None]

# 3) Duplicates hatao (name case-insensitive, pehli entry rakhi jaati hai)
unique = list({p["name"].lower(): p for p in reversed(valid)}.values())

# 4) Category ke hisaab se group
categories = sorted({p["category"] for p in unique})
grouped = {c: [p for p in unique if p["category"] == c] for c in categories}

# 5) Har category ke top 5 mehngay items
top5 = {
    c: sorted(items, key=lambda p: p["price"], reverse=True)[:5]
    for c, items in grouped.items()
}

# 6) Summary dictionary
summary = {
    "total_raw": len(raw_products),
    "total_clean": len(unique),
    "dropped": len(raw_products) - len(unique),
    "categories": {
        c: {
            "count": len(items),
            "avg_price": round(sum(p["price"] for p in items) / len(items), 2),
            "min_price": min(p["price"] for p in items),
            "max_price": max(p["price"] for p in items),
            "top_5": [(p["name"], p["price"]) for p in top5[c]],
        }
        for c, items in grouped.items()
    },
}

In [4]:
#Neat print
print("=" * 58)
print("INVENTORY SUMMARY".center(58))
print("=" * 58)
print(f"Raw: {summary['total_raw']} | Clean: {summary['total_clean']} | Dropped: {summary['dropped']}")

for cat, info in summary["categories"].items():
    print(f"\n{cat}  ({info['count']} items | avg {info['avg_price']:,.2f})")
    print("-" * 58)
    print("\n".join(
        f"  {i}. {name:<20} {price:>10,.2f}"
        for i, (name, price) in enumerate(info["top_5"], start=1)
    ))
print("=" * 58)

                    INVENTORY SUMMARY                     
Raw: 17 | Clean: 13 | Dropped: 4

Electronics  (8 items | avg 309.81)
----------------------------------------------------------
  1. Laptop                 1,899.00
  2. Monitor 24in             180.00
  3. Smart Watch              150.00
  4. Headphones                89.00
  5. Webcam                    59.50

Kitchen  (2 items | avg 15.25)
----------------------------------------------------------
  1. Lunch Box                 18.00
  2. Water Bottle              12.50

Stationery  (2 items | avg 67.50)
----------------------------------------------------------
  1. Notebook A5              120.00
  2. Gel Pen                   15.00

Uncategorized  (1 items | avg 35.00)
----------------------------------------------------------
  1. Desk Lamp                 35.00
